In [1]:
from peft import PeftModel
from transformers import AutoTokenizer, AutoModel
from clalign.alignment import ProteinSeq, AlignmentResult, align_core
from clalign.plm import PLM
from clalign.metrics import f1score

/home/yrh/CLAlign/.conda/envs/CLAlign-Release/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
cd ..

/home/yrh/CLAlign


In [3]:
import torch
import torch.nn.functional as F
import esm
from esm.inverse_folding.util import load_coords, CoordBatchConverter

base_model, alphabet = esm.pretrained.esm_if1_gvp4_t16_142M_UR50()
alphabet.append_eos = True
model = PeftModel.from_pretrained(base_model.encoder, 'src/clalign/CLAlign-ESMIF').cuda()
batch_converter = CoordBatchConverter(alphabet)

/home/yrh/CLAlign/.conda/envs/CLAlign-Release/lib/python3.12/site-packages/esm/pretrained.py:215: UserWarning: Regression weights not found, predicting contacts will not produce correct results.
  warnings.warn(


In [4]:
@torch.no_grad()
def align(s1, s2, seq1, seq2):
    inputs = {x: y.cuda() for x, y in batch_converter.from_lists([s1, s2]).items()}
    embs = F.normalize(model(**inputs).last_hidden_state, dim=-1).cpu().numpy()
    embs = [embs[0, 1:len(seq1) + 1], embs[1, 1:len(seq2) + 1]]
    return align_core(seq1, seq2, embs[0] @ embs[1].T, 0.0)

In [5]:
import numpy as np
from pathlib import Path

def test(data):
    name_list, f_list, tms, tms_ = [], [], [], []
    Path(f'results/{data}/clalign-esmif').mkdir(parents=True, exist_ok=True)
    with open(f'data/{data}_hec.csv') as fp, open(f'results/{data}/clalign-esmif.txt', 'w') as fout:
        fp.readline()
        for line in fp:
            name, f1, f2, seq1, seq2, *_, aln1, aln2 = line.strip().split(',')
            (s1, _), (s2, _) = load_coords(f'data/{data}/{name}/{f1}'), load_coords(f'data/{data}/{name}/{f2}')
            if not np.all(~np.isnan(s1)) or not np.all(~np.isnan(s2)):
                print(name)
            name_list.append(name)
            manual = AlignmentResult(seq1 := ProteinSeq(seq1), seq2 := ProteinSeq(seq2), aln1, aln2)
            aln_res = align(s1, s2, seq1, seq2)
            f_list.append(f_ := f1score(manual, aln_res))
            with open(out_:=(f'results/{data}/clalign-esmif/{name}.txt'), 'w') as faln:
                print('>p1', file=faln)
                print(aln_res.aln1, file=faln)
                print('>p2', file=faln)
                print(aln_res.aln2, file=faln)
            out = !TMalign data/{data}/{name}/{f1} data/{data}/{name}/{f2} -I {out_} -a T
            tms.append(s_:=float(out[17][10:17]))
            out = !TMalign data/{data}/{name}/{f1} data/{data}/{name}/{f2} -i {out_} -a T
            tms_.append(s__:=float(out[17][10:17]))
            print(f'{name}: P: {f_[0]:.3f}, R: {f_[1]:.3f}, F: {f_[2]:.3f}, S: {s_:.5f}, S: {s__:.5f}')
            print(name, f_[0], f_[1], f_[2], s_, file=fout, sep='\t')
        p_, r_, f_ = np.asarray(f_list).mean(axis=0)
        print(f'total: {len(f_list)}, P: {p_:.3f}, R: {r_:.3f}, F: {f_:.3f}, TM-score: {np.mean(tms):.5f}, TM-score: {np.mean(tms_):.5f}')
        return tms

In [6]:
malidup_tms = test('malidup')

d19hca_: P: 0.554, R: 0.554, F: 0.554, S: 0.44708, S: 0.56142
d1a4pa_: P: 0.600, R: 0.600, F: 0.600, S: 0.43994, S: 0.53756
d1a4sa_: P: 0.617, R: 0.617, F: 0.617, S: 0.47899, S: 0.52108
d1a6da1: P: 0.435, R: 0.416, F: 0.425, S: 0.29531, S: 0.44798
d1a8l_1: P: 0.931, R: 0.931, F: 0.931, S: 0.71639, S: 0.73604
d1af2a1: P: 0.595, R: 0.586, F: 0.591, S: 0.52705, S: 0.58075
d1afwb1: P: 0.722, R: 0.722, F: 0.722, S: 0.44072, S: 0.48429
d1ahja_: P: 0.581, R: 0.571, F: 0.576, S: 0.29419, S: 0.35126
d1ahua1: P: 0.505, R: 0.495, F: 0.500, S: 0.37224, S: 0.42530
d1ai3__: P: 0.292, R: 0.382, F: 0.331, S: 0.31905, S: 0.45040
d1aj8a_: P: 0.667, R: 0.667, F: 0.667, S: 0.40489, S: 0.47692
d1ako__: P: 0.691, R: 0.691, F: 0.691, S: 0.47197, S: 0.51269
d1ala___1: P: 0.972, R: 0.972, F: 0.972, S: 0.85174, S: 0.85572
d1ala___2: P: 0.944, R: 0.919, F: 0.932, S: 0.69668, S: 0.70837
d1ala___3: P: 0.887, R: 0.863, F: 0.875, S: 0.67066, S: 0.69763
d1ala___4: P: 0.958, R: 0.945, F: 0.952, S: 0.83057, S: 0.83828


In [7]:
malidup_tms = test('malisam')

d1a05a_d1dgsa3: P: 0.649, R: 0.623, F: 0.636, S: 0.36966, S: 0.48226
d1a05a_d1j71a_: P: 0.305, R: 0.321, F: 0.313, S: 0.30753, S: 0.37788
d1a05a_d1rblm_: P: 0.709, R: 0.709, F: 0.709, S: 0.44210, S: 0.47378
d1a2za_d1ghha_: P: 0.657, R: 0.657, F: 0.657, S: 0.47246, S: 0.52967
d1a2za_d1u9da_: P: 0.679, R: 0.687, F: 0.683, S: 0.45012, S: 0.51141
d1a7j__d1kafa_: P: 0.600, R: 0.696, F: 0.645, S: 0.40351, S: 0.49201
d1a7j__d2if1__: P: 0.520, R: 0.619, F: 0.565, S: 0.35573, S: 0.48987
d1aa7a_d1b68a_: P: 0.132, R: 0.130, F: 0.131, S: 0.29671, S: 0.47766
d1aa7a_d1qkra_: P: 0.269, R: 0.269, F: 0.269, S: 0.26955, S: 0.45414
d1ac5__d1jroa3: P: 0.219, R: 0.194, F: 0.206, S: 0.26481, S: 0.40168
d1ac5__d1vk0a_: P: 0.871, R: 0.897, F: 0.884, S: 0.34490, S: 0.35730
d1adja2d1drw_2: P: 0.553, R: 0.573, F: 0.563, S: 0.42111, S: 0.51243
d1aora2d1dkza2: P: 0.411, R: 0.405, F: 0.408, S: 0.29198, S: 0.42433
d1axn__d1nkta3: P: 0.480, R: 0.480, F: 0.480, S: 0.37747, S: 0.50314
d1axn__d1sf9a_: P: 0.360, R: 0.360